# 02 - Ablate and add the direction
**Paper:** Section 3 (Figures 1 & 3). Bypass refusal on harmful prompts by ablating `r`; induce refusal on harmless prompts by adding it.
Includes a **random-direction control**, which the paper doesn't show but you should: it tells you whether *any* ablation breaks refusal.

In [ ]:
# --- Colab setup: run once per session ---
# 1) Runtime > Change runtime type > T4 GPU (or better)
# 2) Edit REPO to your GitHub URL (if the repo is private, use https://<TOKEN>@github.com/...)
REPO = "https://github.com/YOUR_GITHUB_USERNAME/refusal-direction"

import os
if not os.path.exists("/content/refusal-direction"):
    !git clone -q $REPO /content/refusal-direction
%cd /content/refusal-direction
!git pull -q
%pip install -q -e .
# If pip downgraded numpy and Colab asks you to restart: Runtime > Restart session, then rerun this cell.

In [ ]:
try:  # autoreload picks up edits to src/ without restarting
    %load_ext autoreload
    %autoreload 2
except ModuleNotFoundError:  # Colab's IPython autoreload imports `imp`, which Python 3.12+ removed
    pass
import torch, pandas as pd, matplotlib.pyplot as plt
from refusal_direction.config import Config

# Save artifacts to Google Drive so they survive the Colab session ending.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ARTIFACTS = "/content/drive/MyDrive/refusal-direction/artifacts"
except ImportError:
    ARTIFACTS = "artifacts"
cfg = Config(artifact_dir=ARTIFACTS)
print(cfg)

In [ ]:
from refusal_direction.model import load_model, tokenize_instructions, eoi_positions, generate, last_token_logits
from refusal_direction.data import load_datasets
from refusal_direction.refusal import refusal_token_ids, refusal_score, refusal_rate

model = load_model(cfg)
data = load_datasets(cfg)
tok = lambda instrs, suffix="": tokenize_instructions(model, instrs, cfg.system_prompt, suffix)
refusal_toks = refusal_token_ids(model.tokenizer)
print(model.cfg.n_layers, "layers | d_model", model.cfg.d_model, "| refusal tokens", refusal_toks,
      [model.tokenizer.decode([t]) for t in refusal_toks])

In [ ]:
from refusal_direction import artifacts
from refusal_direction.interventions import get_ablation_hooks, get_addition_hooks
direction, meta = artifacts.load_direction(cfg)
direction = direction.to(model.cfg.device)
layer = meta["layer"]; print(meta)
torch.manual_seed(cfg.seed)
random_dir = torch.randn_like(direction)  # control; scale is irrelevant for ablation

## Harmful prompts: baseline vs. ablation vs. random-direction ablation

In [ ]:
harm = data["harmful_test"]
H = tok(harm)
results = {}
gens = {}
for name, hooks in [("baseline", []), ("ablate_r", get_ablation_hooks(model, direction)),
                    ("ablate_random", get_ablation_hooks(model, random_dir)),
                    ("subtract_r_at_layer", get_addition_hooks(model, direction, layer, coeff=-1.0))]:
    gens[name] = generate(model, *H, fwd_hooks=hooks, max_new_tokens=cfg.max_new_tokens, batch_size=cfg.batch_size)
    results[f"harmful/{name}"] = refusal_rate(gens[name])
results

In [ ]:
for instr, out in list(zip(*(harm, gens["ablate_r"])))[:3]:
    print("PROMPT:", instr, "\n>>>", out.strip()[:300], "\n" + "-" * 80)

## Harmless prompts: baseline vs. adding `r` at its layer

In [ ]:
safe = data["harmless_test"]
S = tok(safe)
for name, hooks in [("baseline", []), ("add_r", get_addition_hooks(model, direction, layer))]:
    gens["harmless_" + name] = generate(model, *S, fwd_hooks=hooks, max_new_tokens=cfg.max_new_tokens, batch_size=cfg.batch_size)
    results[f"harmless/{name}"] = refusal_rate(gens["harmless_" + name])
artifacts.save_json(cfg, "refusal_rates.json", results)
results

In [ ]:
for instr, out in list(zip(*(safe, gens["harmless_add_r"])))[:3]:
    print("PROMPT:", instr, "\n>>>", out.strip()[:300], "\n" + "-" * 80)

In [ ]:
pd.Series(results).plot.barh(figsize=(7, 3), xlim=(0, 1), title="substring refusal rate"); plt.show()

## Questions to write up
1. Compare to the paper's Figure 1. Is the bypass as clean on Qwen2.5-1.5B? Does the random control stay near baseline?
2. Read ~10 ablated completions by hand. Substring matching counts "Sure, here's ... I can't stress enough how illegal this is" as a refusal and vague non-answers as compliance. How far off is the metric? (This is why the paper also uses a safety classifier.)
3. Sweep `coeff` in `get_addition_hooks` (0.5, 1, 2, 4). Where does induced refusal saturate, and when does the output turn to gibberish?